In [25]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler

In [26]:
df = pd.read_csv("../datasets/raw/trek_data.csv", index_col=0)
df.shape

(383, 8)

In [27]:
df.head

<bound method NDFrame.head of                                     Trek              Cost      Time  \
0                 Everest Base Camp Trek  \n$1,420     USD   16 Days   
1           Everest Base Camp Short Trek  \n$1,295     USD   14 Days   
2    Everest Base Camp Heli Shuttle Trek       \n$2000 USD   12 Days   
3            Everest Base Camp Heli Trek  \n$3,300     USD   11 Days   
4     Everest Base Camp Trek for Seniors  \n$1,800     USD   20 Days   
..                                   ...               ...       ...   
378             Ghorepani Poon Hill Trek          $450 USD   10 Days   
379                   Upper Mustang Trek         $2100 USD   17 Days   
380                     Mardi Himal Trek          $590 USD    9 Days   
381             Langtang Valley Trekking          $690 USD   11 Days   
382                 Manaslu Circuit Trek         $1190 USD   17 Days   

        Trip Grade Max Altitude      Accomodation          Best Travel Time  \
0         Moderate       5

In [28]:
df = df.drop(columns=["Contact or Book your Trip"])

In [29]:
df["Trek"] = df["Trek"].str.strip()

In [30]:
#extract cost, keep only float
def clean_cost(val):
    val = str(val)
    val = val.replace('\n', '').replace('USD', '').replace('$', '').replace(',', '').strip()
    return float(val)

df["Cost_USD"] = df["Cost"].apply(clean_cost)
df = df.drop(columns=["Cost"])
print(f"cost range: ${df['Cost_USD'].min():.0f} – ${df['Cost_USD'].max():.0f}")

cost range: $450 – $4200


In [31]:
# some days have trailing space and stuff..
df["Duration_Days"] = (
    df["Time"]
    .str.strip()
    .str.extract(r'(\d+)')[0]
    .astype(int)
)
df = df.drop(columns=["Time"])
print(f"duration range: {df['Duration_Days'].min()} – {df['Duration_Days'].max()} days")

duration range: 5 – 27 days


In [32]:
# extract max altitude
df["Max_Altitude_m"] = (
    df["Max Altitude"]
    .str.replace(',', '', regex=False)
    .str.extract(r'(\d+)')[0]
    .astype(int)
)
df = df.drop(columns=["Max Altitude"])
print(f"altitude range: {df['Max_Altitude_m'].min()} – {df['Max_Altitude_m'].max()} m")

altitude range: 1550 – 6340 m


In [33]:
# trip grade - ordinal encoding 1–8
#    Light < Easy < Easy-Moderate/Light+Moderate 
#    Moderate < Moderate+Demanding/Moderate-Hard 
#    Demanding < Strenuous < Demanding+Challenging
grade_map = {
    'Light':                  1,
    'Easy':                   2,
    'Easy-Moderate':          3,
    'Easy To Moderate':       3,
    'Light+Moderate':         3,
    'Moderate':               4,
    'Moderate+Demanding':     5,
    'Moderate-Hard':          5,
    'Demanding':              6,
    'Strenuous':              7,
    'Demanding+Challenging':  8,
}

In [34]:
df["Trip_Grade_Ordinal"] = df["Trip Grade"].map(grade_map)

unmapped_grades = df[df["Trip_Grade_Ordinal"].isnull()]["Trip Grade"].unique()
if len(unmapped_grades) > 0:
    print(f"WARNING: unmapped grades → {unmapped_grades}")

df = df.drop(columns=["Trip Grade"])
print(f"\nGrade distribution:\n{df['Trip_Grade_Ordinal'].value_counts().sort_index().to_string()}")


Grade distribution:
Trip_Grade_Ordinal
1     5
2    25
3    73
4    92
5    70
6    75
7    23
8    20


In [35]:
# accommodatio, normalize 8 var -> 5 and one hot encoding
accom_map = {
    'Hotel/Guesthouse':    'Guesthouse',
    'Hotel/Guesthouses':   'Guesthouse',
    'Hotel/Guest Houses':  'Guesthouse',
    'Hotel/Teahouse':      'Teahouse',
    'Hotel/Teahouses':     'Teahouse',
    'Hotel/Luxury Lodges': 'LuxuryLodge',
    'Hotel/Lodges':        'Lodge',
    'Teahouses/Lodges':    'TeahouseLodge',
}
df["Accomodation"] = df["Accomodation"].map(accom_map)
unmapped_accom = df[df["Accomodation"].isnull()].shape[0]
if unmapped_accom > 0:
    print(f"WARNING: {unmapped_accom} unmapped accommodation values")

accom_dummies = pd.get_dummies(df["Accomodation"], prefix="Accom").astype(int)
df = pd.concat([df.drop(columns=["Accomodation"]), accom_dummies], axis=1)
print(f"\nAccommodation columns: {list(accom_dummies.columns)}")


Accommodation columns: ['Accom_Guesthouse', 'Accom_Lodge', 'Accom_LuxuryLodge', 'Accom_Teahouse', 'Accom_TeahouseLodge']


In [36]:
#best travel time, normalize 13 var to 5, OHE, fixes typo, trailing period, seps.. dash
def normalize_travel_time(val):
    val = str(val).strip().rstrip('.')      # remove trailing period
    val = val.replace('Setpt', 'Sept')      # fix typo
    val = re.sub(r'\s*-\s*', '-', val)      # "x - y" → "x-y"
    val = re.sub(r'\s*&\s*', ' & ', val)    # normalize spaces around &
    val = re.sub(r'\s+', ' ', val).strip()  # collapse extra whitespace
    return val

travel_category_map = {
    'March-May & Sept-Dec': 'MarMay_SeptDec',
    'April-May & Sept-Nov': 'AprMay_SeptNov',
    'Jan-May & Sept-Dec':   'JanMay_SeptDec',
    'March-May & Sept-Nov': 'MarMay_SeptNov',
    'March-Nov':            'MarNov',
}

df["Best Travel Time"] = (
    df["Best Travel Time"]
    .apply(normalize_travel_time)
    .map(travel_category_map)
)

unmapped_travel = df[df["Best Travel Time"].isnull()].shape[0]
if unmapped_travel > 0:
    print(f"WARNING: {unmapped_travel} unmapped travel time values")

travel_dummies = pd.get_dummies(df["Best Travel Time"], prefix="Travel").astype(int)
df = pd.concat([df.drop(columns=["Best Travel Time"]), travel_dummies], axis=1)
print(f"Travel Time columns: {list(travel_dummies.columns)}")

Travel Time columns: ['Travel_AprMay_SeptNov', 'Travel_JanMay_SeptDec', 'Travel_MarMay_SeptDec', 'Travel_MarMay_SeptNov', 'Travel_MarNov']


In [37]:
# reorder
feature_cols = [c for c in df.columns if c not in ["Trek", "Cost_USD"]]
df = df[["Trek", "Cost_USD"] + feature_cols]

In [38]:
print(f"shape: {df.shape}")
print(f"\ncolumns & types:\n{df.dtypes.to_string()}")
print(f"\nmissing values:\n{df.isnull().sum().to_string()}")
print(f"\nnumeric summary:\n{df.describe().round(2).to_string()}")
print(f"\nfirst 3 rows:\n{df.head(3).to_string()}")

shape: (383, 15)

columns & types:
Trek                         str
Cost_USD                 float64
Duration_Days              int64
Max_Altitude_m             int64
Trip_Grade_Ordinal         int64
Accom_Guesthouse           int64
Accom_Lodge                int64
Accom_LuxuryLodge          int64
Accom_Teahouse             int64
Accom_TeahouseLodge        int64
Travel_AprMay_SeptNov      int64
Travel_JanMay_SeptDec      int64
Travel_MarMay_SeptDec      int64
Travel_MarMay_SeptNov      int64
Travel_MarNov              int64

missing values:
Trek                     0
Cost_USD                 0
Duration_Days            0
Max_Altitude_m           0
Trip_Grade_Ordinal       0
Accom_Guesthouse         0
Accom_Lodge              0
Accom_LuxuryLodge        0
Accom_Teahouse           0
Accom_TeahouseLodge      0
Travel_AprMay_SeptNov    0
Travel_JanMay_SeptDec    0
Travel_MarMay_SeptDec    0
Travel_MarMay_SeptNov    0
Travel_MarNov            0

numeric summary:
       Cost_USD  Duration_Days

In [39]:
## for visualization unscaled.. 
df.to_csv("../datasets/processed/trek_preprocessed_unscaled.csv", index=False)

In [40]:
##f or  kmeans and linear models, scale cont features
## makes no sense to scale binary dummies or target
continuous_cols = ["Duration_Days", "Trip_Grade_Ordinal", "Max_Altitude_m"]
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[continuous_cols] = scaler.fit_transform(df[continuous_cols])

df_scaled.to_csv("../datasets/processed/trek_preprocessed_scaled.csv", index=False)